## Clay encoder (single image)
Opens one AI4Arctic scene, extracts a single 256×256 SAR chip, and encodes it with Clay v1.5's frozen encoder. Only inlines what Clay's encoder actually consumes (`pixels`, `time`, `latlon`, `waves`, `gsd`) — it skips the valid-mask substitution, AMSR2/ERA5 resampling, and CT-label logic from `chip_patch_prep.ipynb`, since the encoder never sees those.

In [ ]:
import math
import re
from datetime import UTC, datetime
from pathlib import Path

import einops
import numpy as np
import torch
import xarray as xr
import yaml
from claymodel.module import ClayMAEModule
from scipy.interpolate import RegularGridInterpolator

### Configuration
`CHIP_ROW`/`CHIP_COL` select which 256×256 tile of the scene to encode (row/col index on the chip grid, not pixel offset). Row 0, col 1 is used here because it's fully free of land/nodata pixels for this scene — pick a different tile and check for NaNs if you swap scenes.

In [ ]:
SCENE_PATH = Path(
    "../../S1A_EW_GRDM_1SDH_20180124T194759_20180124T194859_020301_022AA4_1F75"
    "_icechart_dmi_201801241950_SouthEast_RIC.nc"
)
METADATA_PATH = Path("../../configs/metadata.yaml")
CHECKPOINT_PATH = Path("../../clay-v1.5.ckpt")

CHIP_SIZE = 256
CHIP_ROW, CHIP_COL = 0, 1

### Load the Clay metadata entry for `sentinel-1-ew`
Already committed to `configs/metadata.yaml` (HH index 0, HV index 1; NERSC mean/std from B1; GSD 40m)

In [ ]:
with METADATA_PATH.open() as f:
    metadata = yaml.safe_load(f)

sar_meta = metadata["sentinel-1-ew"]
print(yaml.dump(sar_meta, default_flow_style=False))

### Read one SAR chip from the scene
Opens the NetCDF once, reads HH/HV at native resolution, and slices out a single 256×256 chip. Asserts the chip is nodata-free, since this notebook skips the valid-mask substitution step.

In [ ]:
with xr.open_dataset(SCENE_PATH, engine="netcdf4") as ds:
    r0, c0 = CHIP_ROW * CHIP_SIZE, CHIP_COL * CHIP_SIZE
    r1, c1 = r0 + CHIP_SIZE, c0 + CHIP_SIZE

    sar_chip = np.stack(
        [
            ds["nersc_sar_primary"].values[r0:r1, c0:c1],
            ds["nersc_sar_secondary"].values[r0:r1, c0:c1],
        ]
    ).astype(np.float32)

    gcp_lines = ds["sar_grid_line"].values
    gcp_samps = ds["sar_grid_sample"].values
    gcp_lats = ds["sar_grid_latitude"].values
    gcp_lons = ds["sar_grid_longitude"].values

assert not np.isnan(sar_chip).any(), (
    "Chosen chip has nodata pixels -- pick a different CHIP_ROW/CHIP_COL"
)
print(f"SAR chip shape: {sar_chip.shape}")

### Interpolate the chip's centroid lat/lon from the GCP grid
GCP count read dynamically from the array's own size (mirrors `sar_grid_points`), not hardcoded.

In [ ]:
gcp_side = int(np.sqrt(gcp_lines.size))

# NetCDF storage order is not guaranteed to be row-major.
sort_idx = np.lexsort((gcp_samps, gcp_lines))
lines_2d = gcp_lines[sort_idx].reshape(gcp_side, gcp_side)
samps_2d = gcp_samps[sort_idx].reshape(gcp_side, gcp_side)
lats_2d = gcp_lats[sort_idx].reshape(gcp_side, gcp_side)
lons_2d = gcp_lons[sort_idx].reshape(gcp_side, gcp_side)

kw = dict(method="linear", bounds_error=False, fill_value=None)
interp_lat = RegularGridInterpolator((lines_2d[:, 0], samps_2d[0, :]), lats_2d, **kw)
interp_lon = RegularGridInterpolator((lines_2d[:, 0], samps_2d[0, :]), lons_2d, **kw)

centroid = np.array([[r0 + CHIP_SIZE / 2.0, c0 + CHIP_SIZE / 2.0]])
centroid_lat = float(interp_lat(centroid)[0])
centroid_lon = float(interp_lon(centroid)[0])
print(f"Chip centroid: lat={centroid_lat:.3f}, lon={centroid_lon:.3f}")

### Build Clay's time and latlon positional encodings
Time comes from the acquisition timestamp in the scene filename, not the GCPs.

In [ ]:
match = re.search(r"(\d{8}T\d{6})", SCENE_PATH.name)
acq_dt = datetime.strptime(match.group(1), "%Y%m%dT%H%M%S").replace(tzinfo=UTC)

week = float(acq_dt.isocalendar().week)
hour = acq_dt.hour + acq_dt.minute / 60.0
time_encoding = np.array(
    [
        math.sin(week * 2 * math.pi / 52),
        math.cos(week * 2 * math.pi / 52),
        math.sin(hour * 2 * math.pi / 24),
        math.cos(hour * 2 * math.pi / 24),
    ],
    dtype=np.float32,
)

latlon_encoding = np.array(
    [
        math.sin(centroid_lat * math.pi / 180),
        math.cos(centroid_lat * math.pi / 180),
        math.sin(centroid_lon * math.pi / 180),
        math.cos(centroid_lon * math.pi / 180),
    ],
    dtype=np.float32,
)

print(f"time_encoding:   {time_encoding}")
print(f"latlon_encoding: {latlon_encoding}")

### Load Clay v1.5 (frozen, large)
`model_size="large"` is mandatory — the default `"base"` is 768-dim and silently breaks the feature contract's 1024-dim assumption. `mask_ratio=0.0` and `shuffle=False` per the frozen-encoder invariant.

In [ ]:
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

module = ClayMAEModule.load_from_checkpoint(
    checkpoint_path=str(CHECKPOINT_PATH),
    model_size="large",
    mask_ratio=0.0,
    shuffle=False,
    metadata_path=str(METADATA_PATH),
)
module.eval()
module = module.to(DEVICE)
print(f"Clay loaded on {DEVICE}")

### Encode the chip
Normalises HH/HV with the `sentinel-1-ew` mean/std, builds the datacube, and runs it through the frozen encoder. Index 0 of the output sequence is the class token; indices 1: are the 1024 patch tokens, reshaped to the 32×32 grid.

In [ ]:
sar_mean = torch.tensor(sar_meta["mean"], dtype=torch.float32)
sar_std = torch.tensor(sar_meta["std"], dtype=torch.float32)
sar_norm = (torch.tensor(sar_chip, dtype=torch.float32) - sar_mean[:, None, None]) / sar_std[
    :, None, None
]

datacube = {
    "pixels": sar_norm.unsqueeze(0).to(DEVICE),
    "waves": torch.tensor(sar_meta["wavelengths"], dtype=torch.float32).to(DEVICE),
    "gsd": torch.tensor(float(sar_meta["gsd"]), dtype=torch.float32).to(DEVICE),
    "time": torch.tensor(time_encoding, dtype=torch.float32).unsqueeze(0).to(DEVICE),
    "latlon": torch.tensor(latlon_encoding, dtype=torch.float32).unsqueeze(0).to(DEVICE),
}

with torch.no_grad():
    encoded, *_ = module.model.encoder(datacube)

cls_token = encoded[:, 0, :].cpu().numpy()
patch_tokens = einops.rearrange(encoded[:, 1:, :].cpu().numpy(), "b (h w) d -> b d h w", h=32, w=32)

print(f"Class token shape:  {cls_token.shape}")
print(f"Patch tokens shape: {patch_tokens.shape}")